In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
import os
from pathlib import Path
from time import perf_counter, time
from typing import Any, NamedTuple

import numpy as np

# enable JIT compilation - must be done before loading torch!
os.environ["PYTORCH_JIT"] = "1"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# logging.basicConfig(level=logging.INFO)

In [2]:
import torch
import torchinfo

BATCH_SIZE = 128
TARGET = "OD600"
SPLIT = 0

RUN_NAME = f"{TARGET}-{SPLIT}-More_params"  # | input("enter name for run")

'OD600-0-More_params'

In [3]:
from typing import NamedTuple

import matplotlib.pyplot as plt
import numpy as np
import pandas
from linodenet.models import LinODE, LinODECell, LinODEnet
from linodenet.models.filters import SequentialFilter
from linodenet.projections.functional import skew_symmetric, symmetric
from numpy.typing import NDArray
from pandas import DataFrame, Index, Series, Timedelta, Timestamp
from torch import Tensor, jit, nn, tensor
from torch.optim import SGD, Adam, AdamW
from torch.utils.data import BatchSampler, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary
from tqdm import tqdm, trange

import tsdm
from tsdm.datasets import DATASETS
from tsdm.encoders.functional import time2float
from tsdm.encoders.modular import *
from tsdm.logutils import (
    log_kernel_information,
    log_metrics,
    log_model_state,
    log_optimizer_state,
)
from tsdm.losses import LOSSES
from tsdm.random.samplers import *
from tsdm.tasks import KIWI_FINAL_PRODUCT
from tsdm.util import grad_norm, multi_norm
from tsdm.util.strings import *

np.set_printoptions(4, linewidth=80)

In [4]:
# Disable benchmarking for variable sized input
torch.backends.cudnn.benchmark = True

# Initialize Task

In [5]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32

task = KIWI_FINAL_PRODUCT(
    train_batch_size=BATCH_SIZE,
    eval_batch_size=2048,
    target=TARGET,
)

DATASET = task.dataset
ts = task.timeseries
md = task.metadata
NUM_PTS, NUM_DIM = ts.shape

In [6]:
ts, md = task.splits[SPLIT, "train"]

In [7]:
encoder = ChainedEncoder(
    #
    TensorEncoder(),
    FrameSplitter(["value", "measurement_time", ...]),
    # FrameSplitter([...]),
    TripletEncoder(),
    FrameEncoder(
        Standardizer(),
        index_encoders=MinMaxScaler() @ TimeDeltaEncoder(unit="s"),
    ),
)

ChainedEncoder(
    TensorEncoder(),
    FrameSplitter(
        0: ['value'],
        1: ['measurement_time'],
        2: Ellipsis,
    ),
    TripletEncoder(),
    FrameEncoder(
        column_encoders: Standardizer(axis=None),
        index_encoders: ChainedEncoder(
            MinMaxScaler(axis=None),
            TimeDeltaEncoder(unit='s'),
        ),
    ),
)

In [8]:
demo = ts.reset_index([0, 1], drop=True)

variable,Flow_Air,StirringSpeed,Temperature,Acetate,Base,Cumulated_feed_volume_glucose,Cumulated_feed_volume_medium,DOT,Glucose,OD600,Probe_Volume,pH,Fluo_GFP,InducerConcentration,Volume
measurement_time,,,,,,,,,,,,,,,
0 days 00:00:11,0.0,0.0,36.389999,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
0 days 00:00:21,0.0,0.0,NaN,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,0.0,7.270000,NaN,0.0,NaN
0 days 00:00:26,0.0,100.0,NaN,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
0 days 00:00:27,0.0,100.0,36.130001,NaN,NaN,0.0,0.000000,NaN,NaN,NaN,0.0,NaN,NaN,0.0,NaN
0 days 00:00:36,0.0,100.0,NaN,NaN,NaN,0.0,0.000000,0.000000,NaN,NaN,0.0,7.280000,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 13:27:37,5.0,0.0,NaN,NaN,NaN,1061.0,3787.307373,NaN,NaN,NaN,5500.0,NaN,NaN,0.0,NaN
0 days 13:28:24,5.0,0.0,37.450001,NaN,NaN,1061.0,3787.307373,NaN,NaN,NaN,5500.0,NaN,NaN,0.0,NaN
0 days 13:28:33,5.0,0.0,NaN,NaN,NaN,1061.0,3787.307373,79.459999,NaN,NaN,5500.0,7.320229,NaN,0.0,NaN


In [9]:
encoder.fit(demo)

In [10]:
encoded = encoder.encode(demo)

(tensor([-1.2498, -1.2938, -1.4743,  ...,  1.1824,  0.2823,  6.8684]),
 tensor([0., 0., 0.,  ..., 1., 1., 1.]),
 tensor([[0., 0., 1.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 1., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]))

In [11]:
decoded = encoder.decode(encoded)

variable,Flow_Air,StirringSpeed,Temperature,Acetate,Base,Cumulated_feed_volume_glucose,Cumulated_feed_volume_medium,DOT,Glucose,OD600,Probe_Volume,pH,Fluo_GFP,InducerConcentration,Volume
measurement_time,,,,,,,,,,,,,,,
0 days 00:00:08,0.0,0.0,36.700001,NaN,NaN,0.000000,0.000122,NaN,NaN,NaN,0.0,NaN,NaN,0.000000,NaN
0 days 00:00:11,0.0,0.0,36.389999,NaN,NaN,0.000000,0.000122,NaN,NaN,NaN,0.0,NaN,NaN,0.000000,NaN
0 days 00:00:15,5.0,0.0,36.709999,NaN,NaN,0.000000,0.000122,NaN,NaN,NaN,0.0,NaN,NaN,0.000000,NaN
0 days 00:00:21,0.0,0.0,37.099998,NaN,NaN,0.000000,0.000122,0.000000,NaN,NaN,0.0,7.297619,NaN,0.000000,NaN
0 days 00:00:26,0.0,100.0,NaN,NaN,NaN,0.000000,0.000122,NaN,NaN,NaN,0.0,NaN,NaN,0.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0 days 14:21:42,10.0,0.0,NaN,NaN,NaN,2691.143066,1759.019409,40.185715,NaN,NaN,3600.0,6.745714,NaN,0.757143,NaN
0 days 14:21:44,0.0,0.0,NaN,NaN,NaN,2691.143066,1759.019409,NaN,NaN,NaN,3600.0,NaN,NaN,0.757143,NaN
0 days 14:21:53,0.0,0.0,NaN,NaN,NaN,2691.143066,1759.019409,40.265240,NaN,NaN,3600.0,6.746191,NaN,0.757143,NaN


In [12]:
# pandas.testing.assert_frame_equal(demo, decoded, atol=1e-5)

In [13]:
encoder[-1].column_encoders[0]

Standardizer(axis=())

## Define Batching Function

In [14]:
target_index = ts.columns.get_loc(task.target)
target_encoder = encoder[-1].column_encoders[target_index]

Standardizer(axis=())

In [15]:
class Batch(NamedTuple):
    timeseries: list[Tensor]
    targets: NDArray
    encoded_targets: NDArray


@torch.no_grad()
def mycollate(samples: list[tuple]) -> tuple[list[Tensor], NDArray, NDArray]:
    timeseries = []
    targets = []
    encoded_targets = []

    for idx, (ts_data, md_data), target, originals in samples:
        timeseries.append(encoder.encode(ts_data))
        targets.append(target)
        encoded_targets.append(target_encoder.encode(target))

    # timeseries = torch.cat(timeseries)
    targets = np.stack(targets)
    encoded_targets = torch.tensor(targets)

    return Batch(timeseries, targets, encoded_targets)

In [35]:
dloader = task.get_dataloader((SPLIT, "train"), batch_size=128, shuffle=True)
batch = next(iter(dloader))
sample = batch[0]
batch = mycollate(batch)

Batch(timeseries=[(tensor([-0.4438, -1.4743, -1.2716,  ...,  0.0821,  1.0348,  0.2823]), tensor([0.0419, 0.0419, 0.0419,  ..., 0.9209, 0.9209, 0.9209]), tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.]])), (tensor([ 1.9057, -0.6178, -0.4541,  ...,  1.7808,  0.7396,  2.4914]), tensor([0.0427, 0.0427, 0.0427,  ..., 0.8156, 0.8156, 0.8156]), tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 1., 0.]])), (tensor([-0.4438, -1.4743, -1.2938,  ...,  2.1353, -0.4438, -0.4541]), tensor([0.0431, 0.0431, 0.0431,  ..., 0.8897, 0.8897, 0.8897]), tensor([[1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ...

In [36]:
[len(sample[0]) for sample in batch.timeseries]

[8446,
 7068,
 8141,
 16313,
 7616,
 7616,
 16337,
 8364,
 8692,
 8806,
 7936,
 8452,
 8634,
 8346,
 7775,
 7687,
 6938,
 7847,
 8329,
 7338,
 7228,
 7960,
 8364,
 8692,
 7356,
 7775,
 7622,
 7948,
 16313,
 16283,
 8141,
 7953,
 8290,
 8141,
 8346,
 8251,
 8490,
 7634,
 6938,
 7180,
 8806,
 16604,
 6714,
 7895,
 5784,
 16604,
 6852,
 5784,
 8251,
 7584,
 8320,
 7124,
 6198,
 7983,
 15762,
 16337,
 8692,
 7634,
 7356,
 8634,
 7375,
 7381,
 8692,
 8049,
 7580,
 8329,
 16115,
 7675,
 15828,
 7835,
 7687,
 16880,
 7700,
 8464,
 8092,
 8141,
 6780,
 7948,
 8141,
 6678,
 7883,
 16283,
 7033,
 15762,
 7124,
 7895,
 17066,
 7375,
 8692,
 7829,
 16115,
 7971,
 7138,
 6258,
 5838,
 7960,
 8692,
 7675,
 7823,
 5838,
 6938,
 7907,
 8141,
 8251,
 7369,
 8788,
 8056,
 7080,
 8256,
 8251,
 6852,
 6690,
 6884,
 7723,
 7634,
 8256,
 7829,
 7675,
 7580,
 8214,
 7069,
 7453,
 8692,
 7847,
 8079,
 7948,
 8692,
 8692]

## Initialize Loss & Metrics

In [17]:
LOSS = task.test_metric.to(device=DEVICE)
metrics = {key: jit.script(LOSSES[key]()) for key in ("RMSE", "MSE", "MAE")}

{'RMSE': RecursiveScriptModule(original_name=RMSE),
 'MSE': RecursiveScriptModule(original_name=MSELoss),
 'MAE': RecursiveScriptModule(original_name=L1Loss)}

## Initialize DataLoaders

In [18]:
task.final_product_times

t_min      t_induction            t_max  \
run_id experiment_id                                                      
439    15325          0 days 00:36:11  0 days 09:30:00  0 days 09:30:00   
       15326          0 days 00:36:11  0 days 09:30:00  0 days 09:30:00   
       15327          0 days 00:36:11  0 days 09:00:00  0 days 09:00:00   
       15328          0 days 00:36:11  0 days 09:00:00  0 days 09:00:00   
       15329          0 days 00:36:11  0 days 09:30:00  0 days 09:30:00   
...                               ...              ...              ...   
510    16867          0 days 00:36:21             <NA>  0 days 12:06:59   
       16868          0 days 00:36:21  0 days 10:15:00  0 days 10:15:00   
       16869          0 days 00:36:21             <NA>  0 days 12:06:59   
       16870          0 days 00:36:21  0 days 10:15:00  0 days 10:15:00   
       16871          0 days 00:36:21             <NA>  0 days 12:06:59   

                          target_time  
run_id experiment_id                   
439    15325          0 days 12:44:01  
       15326          0 days 12:44:01  
       15327          0 days 12:44:01  
       15328          0 days 12:44:01  
       15329          0 days 12:44:01  
...                               ...  
510    16867          0 days 13:06:55  
       16868          0 days 13:06:55  
       16869          0 days 13:06:55  
       16870          0 days 13:06:55  
       16871          0 days 13:06:55  

[168 rows x 4 columns]

In [19]:
md["target_value"]

run_id  experiment_id
439     15325            15.316667
        15326            28.916668
        15327            11.883333
        15328            24.900002
        15329            17.033333
                           ...    
510     16867            14.856250
        16868            12.506250
        16869            11.656250
        16870            20.006248
        16871            16.606251
Name: target_value, Length: 126, dtype: float32

In [20]:
TRAINLOADER = task.get_dataloader(
    (SPLIT, "train"),
    batch_size=BATCH_SIZE,
    collate_fn=mycollate,
    pin_memory=True,
    drop_last=True,
    shuffle=True,
    num_workers=8,
    # num_workers=os.cpu_count() // 4,
    persistent_workers=True,
)


EVALLOADER = task.get_dataloader(
    (SPLIT, "test"),
    batch_size=BATCH_SIZE,
    collate_fn=mycollate,
    pin_memory=True,
    drop_last=False,
    shuffle=False,
    num_workers=8,
    # num_workers=os.cpu_count() // 4,
    persistent_workers=True,
)

## Hyperparamters

In [21]:
OPTIMIZER_CONFIG = {
    "__name__": "AdamW",
    "lr": 0.001,
    "betas": (0.9, 0.999),
    "eps": 1e-08,
    "weight_decay": 0.001,
    "amsgrad": False,
}

LR_SCHEDULER_CONFIG = {
    "__name__": "ReduceLROnPlateau",
    "mode": "min",
    # (str) – One of min, max. In min mode, lr will be reduced when the quantity monitored has stopped decreasing; in max mode it will be reduced when the quantity monitored has stopped increasing. Default: ‘min’.
    "factor": 0.1,
    # (float) – Factor by which the learning rate will be reduced. new_lr = lr * factor. Default: 0.1.
    "patience": 10,
    # (int) – Number of epochs with no improvement after which learning rate will be reduced. For example, if patience = 2, then we will ignore the first 2 epochs with no improvement, and will only decrease the LR after the 3rd epoch if the loss still hasn’t improved then. Default: 10.
    "threshold": 0.0001,
    # (float) – Threshold for measuring the new optimum, to only focus on significant changes. Default: 1e-4.
    "threshold_mode": "rel",
    # (str) – One of rel, abs. In rel mode, dynamic_threshold = best * ( 1 + threshold ) in ‘max’ mode or best * ( 1 - threshold ) in min mode. In abs mode, dynamic_threshold = best + threshold in max mode or best - threshold in min mode. Default: ‘rel’.
    "cooldown": 0,
    # (int) – Number of epochs to wait before resuming normal operation after lr has been reduced. Default: 0.
    "min_lr": 1e-08,
    # (float or list) – A scalar or a list of scalars. A lower bound on the learning rate of all param groups or each group respectively. Default: 0.
    "eps": 1e-08,
    # (float) – Minimal decay applied to lr. If the difference between new and old lr is smaller than eps, the update is ignored. Default: 1e-8.
    "verbose": True
    # (bool) – If True, prints a message to stdout for each update. Default: False.
}

{'__name__': 'ReduceLROnPlateau',
 'mode': 'min',
 'factor': 0.1,
 'patience': 10,
 'threshold': 0.0001,
 'threshold_mode': 'rel',
 'cooldown': 0,
 'min_lr': 1e-08,
 'eps': 1e-08,
 'verbose': True}

## Model

In [22]:
from tsdm.models import SetFuncTS

MODEL = SetFuncTS
model = MODEL(17, 1, latent_size=256, dim_keys=128, dim_vals=128, dim_deepset=128)
model.to(device=DEVICE, dtype=DTYPE)
summary(model)

Layer (type:depth-idx)                   Param #
SetFuncTS                                --
├─PositionalEncoder: 1-1                 --
├─DeepSet: 1-2                           --
│    └─MLP: 2-1                          --
│    │    └─Linear: 3-1                  3,200
│    │    └─ReLU: 3-2                    --
│    │    └─Dropout: 3-3                 --
│    │    └─Linear: 3-4                  16,512
│    │    └─ReLU: 3-5                    --
│    │    └─Dropout: 3-6                 --
│    │    └─Linear: 3-7                  16,512
│    └─MLP: 2-2                          --
│    │    └─Linear: 3-8                  16,512
│    │    └─ReLU: 3-9                    --
│    │    └─Dropout: 3-10                --
│    │    └─Linear: 3-11                 16,512
│    │    └─ReLU: 3-12                   --
│    │    └─Dropout: 3-13                --
│    │    └─Linear: 3-14                 16,512
├─MLP: 1-3                               --
│    └─Linear: 2-3                       3,200
│

### Warmup - test forward / backward pass

In [23]:
batch = next(iter(TRAINLOADER))
y = model.forward_batch(batch.timeseries)
torch.linalg.norm(y).backward()
model.zero_grad()

## Initalize Optimizer

In [24]:
from tsdm.optimizers import LR_SCHEDULERS, OPTIMIZERS
from tsdm.util import initialize_from

In [25]:
OPTIMIZER_CONFIG |= {"params": model.parameters()}
optimizer = initialize_from(OPTIMIZERS, **OPTIMIZER_CONFIG)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    eps: 1e-08
    lr: 0.001
    weight_decay: 0.001
)

## Load Existing Model instead

In [26]:
# # load existing model & optimizer
# from pathlib import Path

# model_path = Path(
#     "/home/rscholz/Projects/KIWI/tsdm/examples/"
#     "checkpoints/SetFuncTS/KIWI_RUNS/OD600-0-More_params/2022-02-07T00:55:25"
# )
# model_name = "SetFuncTS"
# optim_name = "AdamW"
# model_versions = {}
# optim_versions = {}

# for file in model_path.iterdir():
#     name, version = file.stem.split("-")
#     if name == model_name:
#         model_versions[int(version)] = file
#     if name == optim_name:
#         optim_versions[int(version)] = file

# MODEL = model_versions[max(model_versions)]
# model = jit.load(MODEL)
# MODEL = SetFuncTS
# OPTIM = optim_versions[max(optim_versions)]
# optim = torch.load(OPTIM)

# epoch = optim["epoch"]
# batch_num = optim["batch"]
# optim = optim["optimizer"]

## Logging Utilities

In [27]:
from tsdm.logutils import compute_metrics

# def log_all(i, model, writer, optimizer):
#     kernel = model.system.kernel.clone().detach().cpu()
#     log_kernel_information(i, writer, kernel, histograms=True)
#     log_optimizer_state(i, writer, optimizer, histograms=True)


def log_hparams(i, writer, *, metric_dict, hparam_dict):
    hparam_dict |= {"epoch": i}
    metric_dict = add_prefix(metric_dict, "hparam")
    writer.add_hparams(hparam_dict=hparam_dict, metric_dict=metric_dict)

In [28]:
RUN_START = tsdm.util.now()
CHECKPOINTDIR = Path(
    f"checkpoints/{MODEL.__name__}/{DATASET.name}/{RUN_NAME}/{RUN_START}"
)
CHECKPOINTDIR.mkdir(parents=True, exist_ok=True)
LOGGING_DIR = f"runs/{MODEL.__name__}/{DATASET.name}/{RUN_NAME}/{RUN_START}"
writer = SummaryWriter(LOGGING_DIR)

# Training

In [29]:
@torch.no_grad()
def get_all_predictions(model, dataloader):
    ys = []
    yhats = []
    for batch in tqdm(dataloader, leave=False):
        # ts = batch.timeseries
        # inputs = [(t.to(device=DEVICE),v.to(device=DEVICE), m.to(device=DEVICE)) for t,v,m in ts]
        # yhats.append(model.batch_forward(inputs))
        yhats.append(model.forward_batch(batch.timeseries))
        ys.append(batch.encoded_targets.to(device=DEVICE))
    y = torch.cat(ys).cpu().numpy()
    yhat = torch.cat(yhats).cpu().numpy()
    y = torch.tensor(target_encoder.decode(y))
    yhat = torch.tensor(target_encoder.decode(yhat))
    return y, yhat

In [30]:
i = 0 #batch_num
epoch = 0# epoch

with torch.no_grad():
    for key, dloader in {"train": TRAINLOADER, "test": EVALLOADER}.items():
        y, ŷ = get_all_predictions(model, dloader)
        assert torch.isfinite(y).all()
        log_metrics(epoch, writer=writer, metrics=metrics, targets=y, predics=ŷ, prefix=key)

In [ ]:
for epoch in (epochs := range(epoch, epoch + 1000)):
    if epoch == 1000:
        for g in optimizer.param_groups:
            g["lr"] = 0.0001
    batching_time = perf_counter()
    for batch in (batches := tqdm(TRAINLOADER, leave=False)):
        batching_time = perf_counter() - batching_time
        i += 1
        # Optimization step
        model.zero_grad(set_to_none=True)
        targets = batch.encoded_targets.to(device=DEVICE)

        # forward
        forward_time = perf_counter()
        predics = model.forward_batch(batch.timeseries)
        loss = LOSS(targets, predics)
        forward_time = perf_counter() - forward_time

        # backward
        backward_time = perf_counter()
        loss.backward()
        backward_time = perf_counter() - backward_time

        # step
        optimizer.step()

        # batch logging
        with torch.no_grad():
            logging_time = time()
            if torch.any(~torch.isfinite(loss)):
                raise RuntimeError("NaN/INF-value encountered!!")

            log_metrics(
                i,
                writer=writer,
                metrics=metrics,
                targets=targets.clone(),
                predics=predics.clone(),
                prefix="batch",
            )
            log_optimizer_state(i, writer=writer, optimizer=optimizer, prefix="batch")

            # lval = loss.clone().detach().cpu().numpy()
            # gval = grad_norm(list(model.parameters())).clone().detach().cpu().numpy()
            logging_time = time() - logging_time

        batches.set_postfix(
            # loss=f"{lval:.2e}",
            # gnorm=f"{gval:.2e}",
            epoch=epoch,
            Δt_forward=f"{forward_time:.1f}",
            Δt_backward=f"{backward_time:.1f}",
            Δt_logging=f"{logging_time:.1f}",
            Δt_batching=f"{batching_time:.1f}",
        )
        batching_time = perf_counter()

    with torch.no_grad():
        # end-of-epoch logging
        for key, dloader in {"train": TRAINLOADER, "test": EVALLOADER}.items():
            y, ŷ = get_all_predictions(model, dloader)
            assert torch.isfinite(y).all()
            log_metrics(
                epoch, writer=writer, metrics=metrics, targets=y, predics=ŷ, prefix=key
            )

        # Model Checkpoint
        torch.jit.save(model, CHECKPOINTDIR.joinpath(f"{MODEL.__name__}-{epoch}"))
        torch.save(
            {
                "optimizer": optimizer,
                "optimizer_state": optimizer.state_dict(),
                "epoch": epoch,
                "batch": i,
            },
            CHECKPOINTDIR.joinpath(f"{optimizer.__class__.__name__}-{epoch}"),
        )

  0%|                                                    | 0/98 [00:00<?, ?it/s]/home/rscholz/Projects/KIWI/tsdm/tsdm/logutils/_logutils.py:116: RuntimeWarning: Optimizer logging only works with ADAM currently
  warnings.warn(
  0%|                                                    | 0/30 [00:00<?, ?it/s]

In [ ]:
predics

In [ ]:
raise StopIteration

# Post Training Analysis

# Profiling

In [ ]:
from torch.profiler import ProfilerActivity, profile, record_function

In [ ]:
with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    model(times, inputs)

In [ ]:
print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))

In [ ]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))